In [1]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torchvision.models as models
import torch.nn as nn
import torch
import torch.optim as optim
from tqdm import tqdm   

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # resize images to 224x224
    transforms.ToTensor(),          # convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])  # standard ImageNet normalization
])


In [3]:
num_classes = 4
# Load datasets
train_dataset = datasets.ImageFolder(root="../data/Cancer_Dataset/train/kidney", transform=transform)
val_dataset = datasets.ImageFolder(root="../data/Cancer_Dataset/val/kidney", transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [4]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

/gpfs/helios/home/naveenkumar/.conda/envs/revvity/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/gpfs/helios/home/naveenkumar/.conda/envs/revvity/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


cuda


In [5]:
for epoch in range(10):  # example 5 epochs
    print(f"\nEpoch {epoch+1} starting...")
    model.train()
    running_loss = 0.0

    # Wrap dataloader with tqdm
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", unit="batch"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")



Epoch 1 starting...


Epoch 1: 100%|██████████| 273/273 [00:52<00:00,  5.15batch/s]


Epoch 1, Loss: 0.1749

Epoch 2 starting...


Epoch 2: 100%|██████████| 273/273 [00:44<00:00,  6.11batch/s]


Epoch 2, Loss: 0.0421

Epoch 3 starting...


...

Epoch 8, Loss: 0.0001

Epoch 9 starting...


Epoch 9: 100%|██████████| 273/273 [00:43<00:00,  6.27batch/s]


Epoch 9, Loss: 0.0596

Epoch 10 starting...


Epoch 10: 100%|██████████| 273/273 [00:43<00:00,  6.25batch/s]

Epoch 10, Loss: 0.0257


In [6]:
# Validation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Validating", unit="batch"):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Validation Accuracy: {100 * correct / total:.2f}%')


Validating: 100%|██████████| 117/117 [00:20<00:00,  5.75batch/s]

Validation Accuracy: 99.87%


In [ ]:
# Save model
torch.save(model.state_dict(), "kidney_model.pth")